# 03 - Model Training

This notebook mirrors the training pipeline in `src/train.py` but runs it interactively.

Steps:
- Load configuration and data.
- Build features and perform a time-based train/validation split.
- Train an XGBoost regressor with early stopping.
- Evaluate baseline vs model (MAE, RMSE).


In [ ]:
import numpy as np

from src.config import get_default_config, ensure_project_dirs
from src.preprocessing import prepare_base_dataframe
from src.features import build_features, get_feature_columns
from src.model import build_xgb_model
from src.evaluate import evaluate_predictions
from src.utils import get_logger, train_val_time_split

logger = get_logger()
cfg = get_default_config()
ensure_project_dirs(cfg.paths)

base_df = prepare_base_dataframe(cfg.paths, cfg.subset)
feat_df = build_features(base_df)
feature_cols = get_feature_columns(feat_df)

train_mask, val_mask = train_val_time_split(feat_df["date"].values, cfg.split.validation_days)
train_df = feat_df.loc[train_mask].copy()
val_df = feat_df.loc[val_mask].copy()

X_train = train_df[feature_cols].astype(np.float32).values
y_train = train_df["sales"].astype(np.float32).values
X_val = val_df[feature_cols].astype(np.float32).values
y_val = val_df["sales"].astype(np.float32).values

print("Train size:", X_train.shape, "Val size:", X_val.shape)

In [ ]:
# Baseline (lag-1)
baseline_pred = val_df["lag_1"].values
baseline_metrics = evaluate_predictions(y_val, baseline_pred)
baseline_metrics

In [ ]:
# Train XGBoost model
model = build_xgb_model(cfg.model)
model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_metric="rmse",
    early_stopping_rounds=cfg.model.early_stopping_rounds,
    verbose=True,
)

val_pred = model.predict(X_val)
model_metrics = evaluate_predictions(y_val, val_pred)
model_metrics